
**Created by:**

__Viktor Varga__

<br>
<img src="../_files/nb_im/ik_logo.png" style="display:inline-block" width='40%'>
<hr>

# **NumPy Array Programming Guide** – Chapter 5


## Referencing array elements – Boolean masks (Advanced indexing 2.)


In the previous chapter we saw that multiple arbitrary elements of arrays can be accessed or overwritten using *advanced indexing*. We used integer arrays (or lists) for indexing, where the elements of those arrays specify the indices of the referenced elements, and we used separate index arrays per axis.

However, *advanced indexing* also includes another option for **referencing elements**: instead of integer indices, we can use **boolean arrays/lists (masks) of a suitable shape**.

First, consider the one‑dimensional example. Let’s create an array whose elements we will reference!


In [ ]:
import numpy as np

a = np.arange(6, dtype=np.int32)

print("The 'a' array:", a)

The 'a' array: [0 1 2 3 4 5]


The simplest case of mask-based indexing is when the array and the mask have the same shape. Let’s create a boolean array with the same shape!


In [ ]:
mask_arr1 = np.array([False, True, False, True, True, False])

print("The shape of the 'a' array:", a.shape)
print("The shape of the 'mask_arr1' array:", mask_arr1.shape)

print("\nIndexing with bool array (mask) -> a[mask_arr1]:", a[mask_arr1])

The shape of the 'a' array: (6,)
The shape of the 'mask_arr1' array: (6,)

Indexing with bool array (mask) -> a[mask_arr1]: [1 3 4]


The array `mask_arr1` can be used to reference elements of `a` as follows:
```
a[mask_arr1]
```
The result of this expression is a new array that **contains those elements of `a` for which the boolean value at the corresponding position in `mask_arr1` is true.** Naturally, the exception is when the indexing expression appears on the left-hand side of an assignment.

As shown in the cell below, in the simple one‑dimensional case the number of `True` values in the mask equals the number of elements in the result array. We can count the `True` values, among other ways, with `np.count_nonzero()`. The name may need explanation: NumPy represents the boolean literals `False` and `True` (dtype `np.bool_`) as the numeric values 0 and 1 by default. `np.count_nonzero()` counts the non‑zero elements of an array (or its slices); in a boolean array this therefore returns the number of `True` values:


In [ ]:
print("\n-- The number of True (nonzero) elements in the mask:")
print("np.count_nonzero(mask_arr1):", np.count_nonzero(mask_arr1))

print("\n-- The shape of the result array from using the mask:")
print("a[mask_arr1].shape:", a[mask_arr1].shape)


-- The number of True (nonzero) elements in the mask:
np.count_nonzero(mask_arr1): 3

-- The shape of the result array from using the mask:
a[mask_arr1].shape: (3,)


Instead of `np.count_nonzero()`, we can also use `np.sum()` to count true values, but the latter is a less efficient choice for this purpose.

As with integer-index advanced indexing, we are not required to use an array for indexing. A list or tuple can also be used, but when indexing multiple axes simultaneously it matters how these are applied.

In the cell below you can see that masking with a list of all `False` yields an empty result, while masking with a list of all `True` yields a copy of the original array.


In [ ]:
a = np.arange(6, dtype=np.int32)

print("-- Using different 1D masks to extract elements of the 'a' array:")
print("a[[False, True, False, True, True, False]]:", a[[False, True, False, True, True, False]])
print("a[[False, False, False, False, False, False]]:", a[[False, False, False, False, False, False]])
print("a[[True, True, True, True, True, True]]:", a[[True, True, True, True, True, True]])


-- Using different 1D masks to extract elements of the 'a' array:
a[[False, True, False, True, True, False]]: [1 3 4]
a[[False, False, False, False, False, False]]: []
a[[True, True, True, True, True, True]]: [0 1 2 3 4 5]


Do not forget that **this is still *advanced indexing*, so the referenced elements are placed into a new array, and writing to it does not affect the original array.**


In [ ]:
a = np.arange(6, dtype=np.int32)
print("The 'a' array:", a)

print("\n-- Creating the 'a2' array by masking the 'a' array:")
a2 = a[mask_arr1]
print("The 'a2' array:", a2)

print("\n-- Modifying the 'a2' array:")
a2[1] = 99

print("The modified 'a2' array:", a2)

print("\n-- The original 'a' array was not modified, as 'a2' is not a view of it, but a new array:")
print("The 'a' array, after 'a2' was modified:", a)

The 'a' array: [0 1 2 3 4 5]

-- Creating the 'a2' array by masking the 'a' array:
The 'a2' array: [1 3 4]

-- Modifying the 'a2' array:
The modified 'a2' array: [ 1 99  4]

-- The original 'a' array was not modified, as 'a2' is not a view of it, but a new array:
The 'a' array, after 'a2' was modified: [0 1 2 3 4 5]


In the next cell, advanced indexing with the mask is shown on the left side of the assignment. The elements referenced in this way can be overwritten with the elements on the right side of the assignment. In the one-dimensional case, the number of elements referenced on the left side (i.e., the number of True values in the mask) must match the number of values to be entered on the right side, OR, there can be exactly one value on the right side. If the latter is true, we write the same value into the indices referenced by the array mask. This is in line with the rules of broadcasting.

In [ ]:
a = np.arange(6, dtype=np.int32)
print("The 'a' array:", a)

a[mask_arr1] = 99         # wrapping the Python int into an (1,) shaped ndarray and broadcasting it to match mask shape

print("\nThe 'a' array after writing with mask:", a)

a[mask_arr1] = 42,77,74
#a[mask_arr1] = [42,77,74]                                 # also correct
#a[mask_arr1] = np.array([42,77,74], dtype=np.int32)       # also correct

print("The 'a' array after writing with mask, again:", a)

The 'a' array: [0 1 2 3 4 5]

The 'a' array after writing with mask: [ 0 99  2 99 99  5]
The 'a' array after writing with mask, again: [ 0 42  2 77 74  5]


Next, consider multidimensional cases. In the cell below we use a 2D mask of the same shape as a 2D array:


In [ ]:
arr2d = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 'arr2d' array:\n", arr2d)
print("The shape of the 'arr2d' array:", arr2d.shape)

print("\n-- Creating a 2D mask manually with a shape similar to the array:")
mask2d = np.array([[False, True, False], [True, True, False]])
print("The 'mask2d' array:\n", mask2d)
print("The shape of the 'mask2d' array:", mask2d.shape)

print("\n-- Using this mask to extract selected elements into a new array:")
print("arr2d[mask2d]:", arr2d[mask2d])
print("The shape of 'arr2d[mask2d]':", arr2d[mask2d].shape)


The 'arr2d' array:
 [[0 1 2]
 [3 4 5]]
The shape of the 'arr2d' array: (2, 3)

-- Creating a 2D mask manually with a shape similar to the array:
The 'mask2d' array:
 [[False  True False]
 [ True  True False]]
The shape of the 'mask2d' array: (2, 3)

-- Using this mask to extract selected elements into a new array:
arr2d[mask2d]: [1 3 4]
The shape of 'arr2d[mask2d]': (3,)


For multidimensional arrays, it is not always necessary for the array and the mask to have the same shape. It is enough, if their shapes can be matched by broadcasting. As a result, **we can apply masks only along certain axes**, as shown in the second and third rows of the figure below. When indexing a 2D array (matrix), these masks can naturally reference the rows or columns of the matrix.
<br>$\ $</br>

<img src='../_files/nb_im/numpy_basics-adv_idx_mask.png' width='60%'>

The code cell below shows examples of referencing full rows and columns using masks applied along a single axis:


In [ ]:
arr2d = np.arange(6, dtype=np.int32).reshape((2,3))
print("The 'arr2d' array:\n", arr2d)
print("The shape of the 'arr2d' array:", arr2d.shape)

print("\n-- Creating separate masks for the two axes (one for rows, one for columns):")

# MASK for axis#0
mask_br0 = np.array([False, True])
print("The 'mask_br0' array:\n", mask_br0)
print("The shape of the 'mask_br0' array:", mask_br0.shape)

# MASK for axis#1
mask_br1 = np.array([True, False, True])
print("\nThe 'mask_br1' array:\n", mask_br1)
print("The shape of the 'mask_br1' array:", mask_br1.shape)

# mask along axis#0, basic indexing along axis#1
print("\n-- Masking rows:")
print("arr2d[mask_br0,:]:", arr2d[mask_br0,:])   # equivalent to arr2d[mask_br0]
print("arr2d[mask_br0,:].shape:", arr2d[mask_br0,:].shape)

# basic indexing along axis#0,  mask along axis#1
print("\n-- Masking columns:")
print("arr2d[:,mask_br1]:\n", arr2d[:,mask_br1])
print("arr2d[:,mask_br1].shape:", arr2d[:,mask_br1].shape)

# AND relation of two independent masks along the two axes
print("\n-- Masking columns AND rows at the same time:")
print("arr2d[mask_br0, mask_br1]:", arr2d[mask_br0, mask_br1])


The 'arr2d' array:
 [[0 1 2]
 [3 4 5]]
The shape of the 'arr2d' array: (2, 3)

-- Creating separate masks for the two axes (one for rows, one for columns):
The 'mask_br0' array:
 [False  True]
The shape of the 'mask_br0' array: (2,)

The 'mask_br1' array:
 [ True False  True]
The shape of the 'mask_br1' array: (3,)

-- Masking rows:
arr2d[mask_br0,:]: [[3 4 5]]
arr2d[mask_br0,:].shape: (1, 3)

-- Masking columns:
arr2d[:,mask_br1]:
 [[0 2]
 [3 5]]
arr2d[:,mask_br1].shape: (2, 2)

-- Masking columns AND rows at the same time:
arr2d[mask_br0, mask_br1]: [3 5]


We can also mask multiple adjacent axes at once using a common multidimensional mask. If we want to apply a common mask across multiple non‑adjacent axes at the same time, we first need to permute axes (e.g. `np.swapaxes()`, `np.transpose()`, `ndarray.T`).

Different indexing methods (scalar indices, *basic indexing*, integer and mask-based *advanced indexing*) can be mixed across axes, and broadcasting rules apply here as well. More information and examples are in the documentation: https://numpy.org/devdocs/user/basics.indexing.html#advanced-indexing


## Elementwise boolean operations


In the first half of the chapter we reviewed the details of indexing with boolean arrays, i.e. masks. It is uncommon, however, to define masks by manually specifying boolean values element by element. Much more often, **masks are obtained as the result of some boolean operation**, and we then use these `np.bool_` result arrays for mask-based indexing.

In the cell below we review a few operations that map numeric types to boolean values:


In [ ]:
arr1 = np.arange(7)
arr2 = arr1[::-1]

print("The 'arr1' array:", arr1)
print("The 'arr2' array:", arr2)

print("\n-- 'Numeric x numeric -> bool' binary operators:")
print("arr1 == arr2 is:", arr1 == arr2)   # np.equal()
print("arr1 != arr2 is:", arr1 != arr2)   # np.not_equal()
print("\narr1 < arr2 is:", arr1 < arr2)   # np.less()
print("arr1 > arr2 is:", arr1 > arr2)   # np.greater()
print("arr1 <= arr2 is:", arr1 <= arr2)   # np.less_equal()
print("arr1 >= arr2 is:", arr1 >= arr2)   # np.greater_equal()
print("\nnp.isclose(narr1, arr2):", np.isclose(arr1, arr2, rtol=0.001, atol=1e-6))

The 'arr1' array: [0 1 2 3 4 5 6]
The 'arr2' array: [6 5 4 3 2 1 0]

-- 'Numeric x numeric -> bool' binary operators:
arr1 == arr2 is: [False False False  True False False False]
arr1 != arr2 is: [ True  True  True False  True  True  True]

arr1 < arr2 is: [ True  True  True False False False False]
arr1 > arr2 is: [False False False False  True  True  True]
arr1 <= arr2 is: [ True  True  True  True False False False]
arr1 >= arr2 is: [False False False  True  True  True  True]

np.isclose(narr1, arr2): [False False False  True False False False]


Broadcasting rules apply here as well:


In [ ]:
arr2d = np.arange(6).reshape((2,3))
print("The 'arr2d' array:\n", arr2d)

print("\narr2d < 4 is:\n", arr2d < 4)
print("arr2d < [3,2,2] is:\n", arr2d < [3,2,2])          # (2,3) vs (3,) -> broadcasting OK
# print("arr2d < [3,2] is:\n", arr2d < [3,2])            # (2,3) vs (2,) -> broadcasting INCORRECT; ValueError: operands could not be broadcast together with shapes (2,3) (2,)
print("arr2d < [[1], [3]] is:\n", arr2d < [[1], [3]])    # (2,3) vs (2, 1) -> broadcasting OK;

The 'arr2d' array:
 [[0 1 2]
 [3 4 5]]

arr2d < 4 is:
 [[ True  True  True]
 [ True False False]]
arr2d < [3,2,2] is:
 [[ True  True False]
 [False False False]]
arr2d < [[1], [3]] is:
 [[ True False False]
 [False False False]]


In the next cell we see elementwise operations that are not only boolean-valued but also expect boolean operands. Elementwise negation (NOT), conjunction (AND), disjunction (OR), and exclusive-or (XOR) are such operations:


In [ ]:
mask1 = np.array([False, True, False, True, False, False])
mask2 = np.array([True, True, False, True, True, False])

print("The 'mask1' array:\n", mask1)
print("The 'mask2' array:\n", mask2)

print("\n-- The only unary logical operator, NOT:")
print("~mask1:", ~mask1)   # np.logical_not()

print("\n-- The three binary logical operators, AND, OR, XOR:")
print("mask1 & mask2 is:", mask1 & mask2)   # np.logical_and()
print("mask1 | mask2 is:", mask1 | mask2)   # np.logical_or()
print("mask1 ^ mask2 is:", mask1 ^ mask2)   # np.logical_xor()


The 'mask1' array:
 [False  True False  True False False]
The 'mask2' array:
 [ True  True False  True  True False]

-- The only unary logical operator, NOT:
~mask1: [ True False  True False  True  True]

-- The three binary logical operators, AND, OR, XOR:
mask1 & mask2 is: [False  True False  True False False]
mask1 | mask2 is: [ True  True False  True  True False]
mask1 ^ mask2 is: [ True False False False  True False]


## *Vector → scalar* (collapse) operations for boolean dtype


In earlier chapters we already considered some *vector → scalar* operations. Given entire arrays or their slices as input, they return a single scalar (or one scalar per slice). Examples include `np.sum()` and `np.mean()`. Just as `np.sum()` is a reduction using the addition operator, NumPy also provides functions for reductions using logical operators.

The `np.any()` and `np.all()` functions shown in the next cell correspond to reductions with logical OR and logical AND, respectively. In other words, `np.any()` returns whether there exists at least one true value in the input array/slice, while `np.all()` is true only if every element in the array/slice is true.


In [ ]:
mask2d = np.array([[False, True, False], [True, True, False]])
print("The 'mask2d' array:\n", mask2d)

print("\n-- Finding whether 2D mask has only / any True elements:")
print("np.all(mask2d):", np.all(mask2d))
print("np.any(mask2d):", np.any(mask2d))

The 'mask2d' array:
 [[False  True False]
 [ True  True False]]

-- Finding whether 2D mask has only / any True elements:
np.all(mask2d): False
np.any(mask2d): True


As with `np.sum()` and `np.mean()`, the `axis` parameter can be used here as well to perform the operation independently on slices enumerated along a given axis.


In [ ]:
print("The 'mask2d' array:\n", mask2d)

print("\n-- Finding rows and columns with only True elements:")
print("np.all(mask2d, axis=0):", np.all(mask2d, axis=0))
print("np.all(mask2d, axis=1):", np.all(mask2d, axis=1))

print("\n-- Finding rows and columns with any True elements:")
print("np.any(mask2d, axis=0):", np.any(mask2d, axis=0))
print("np.any(mask2d, axis=1):", np.any(mask2d, axis=1))


The 'mask2d' array:
 [[False  True False]
 [ True  True False]]

-- Finding rows and columns with only True elements:
np.all(mask2d, axis=0): [False  True False]
np.all(mask2d, axis=1): [False False]

-- Finding rows and columns with any True elements:
np.any(mask2d, axis=0): [ True  True False]
np.any(mask2d, axis=1): [ True  True]


## Converting between *mask ↔ coordinate arrays*


The two forms of *advanced indexing* are (1) referencing with integer index (coordinate) arrays, and (2) masking with boolean arrays. In practice, either technique can replace the other; often the deciding factor is which input or output format is more convenient to process or produce.

It is not uncommon to switch between the two formats in the middle of a workflow. A good example is the following: we work with images represented as 2D arrays, where the array values represent pixel brightnesses. If we want to extract the coordinates of pixels whose brightness is above a certain threshold, we need a *mask → coordinate array* conversion.

The essence of the conversion is that **the indices listed in the coordinate array are those where True values are found in the mask**. One of the functions that implements this conversion will be `np.where()`. The `np.where(condition)` function expects an array of type `np.bool_` in the `condition` parameter, i.e., a mask. The function returns the indices of the true values found in the mask.

Let's first consider the simple, one-dimensional case:

In [ ]:
mask1d = np.array([False, True, True, False, True, False])
print("The 'mask1d' array:", mask1d)

idxs_tup = np.where(mask1d)

print("\n-- Extracting indices of True elements in 1D mask with np.where():")
print("np.where(mask1d):", idxs_tup)
print("type of np.where(mask1d):", type(idxs_tup))

idxs, = np.where(mask1d)    # extracting single array from tuple with a length of 1
# idxs = np.where(mask1d)[0]

print("\nnp.where(mask1d), array unpacked from the tuple:", idxs)

The 'mask1d' array: [False  True  True False  True False]

-- Extracting indices of True elements in 1D mask with np.where():
np.where(mask1d): (array([1, 2, 4]),)
type of np.where(mask1d): <class 'tuple'>

np.where(mask1d), array unpacked from the tuple: [1 2 4]


As we see in the cell above, **`np.where()` does not simply return an integer array of indices; it places it into a one‑element tuple.** This is easier to understand if we look at the case where the input mask is already two‑dimensional:


In [ ]:
mask2d = mask1d.reshape(2,3)
print("The 'mask2d' array:\n", mask2d)

print("\n-- Extracting indices of True elements in 2D mask with np.where():")
idxs_tup = np.where(mask2d)
print("np.where(mask2d):", idxs_tup)

The 'mask2d' array:
 [[False  True  True]
 [False  True False]]

-- Extracting indices of True elements in 2D mask with np.where():
np.where(mask2d): (array([0, 0, 1]), array([1, 2, 1]))


**If the input array is two‑dimensional, the returned tuple has length 2 and contains two integer arrays of the same length.** This is the standard format for multidimensional indexing in NumPy: we separate the indices of the referenced elements into 1D arrays per axis. Thus, **the result of `np.where()` can be used directly to index the input mask, or any other array with the same shape.**

`np.argwhere()` is similar to `np.where()`, but it returns the same indices in a different format:


In [ ]:
print("-- Extracting indices of True elements in 1D mask with np.where() and np.argwhere():")
print("\nnp.where(mask1d):", np.where(mask1d))
print("np.argwhere(mask1d):\n", np.argwhere(mask1d))
print("Shape of np.argwhere(mask1d):", np.argwhere(mask1d).shape)

print("\n-- Extracting indices of True elements in 2D mask with np.where() and np.argwhere():")
print("\nnp.where(mask2d):", np.where(mask2d))
print("np.argwhere(mask2d):", np.argwhere(mask2d))
print("Shape of np.argwhere(mask2d):", np.argwhere(mask2d).shape)

-- Extracting indices of True elements in 1D mask with np.where() and np.argwhere():

np.where(mask1d): (array([1, 2, 4]),)
np.argwhere(mask1d):
 [[1]
 [2]
 [4]]
Shape of np.argwhere(mask1d): (3, 1)

-- Extracting indices of True elements in 2D mask with np.where() and np.argwhere():

np.where(mask2d): (array([0, 0, 1]), array([1, 2, 1]))
np.argwhere(mask2d): [[0 1]
 [0 2]
 [1 1]]
Shape of np.argwhere(mask2d): (3, 2)


While `np.where()` separates the coordinates of `True` elements by axis and returns 1D arrays packed in a tuple, `np.argwhere()` returns a single 2D array whose column vectors correspond to the arrays returned by `np.where()`. The figure below illustrates the difference between the two formats:

<br>$\ $</br>

<img src='../_files/nb_im/numpy_basics-where_argwhere.png' width='80%'>

In the next cell you can see that the result of `np.where()` is directly usable for indexing:

```
mask2d[np.where(mask2d)]
```

while the result of `np.argwhere()` can be used as follows:

```
idxs = np.argwhere(mask2d)
mask2d[idxs[:, 0], idxs[:, 1]]
```

In some cases the latter form is preferable, since it avoids mixing arrays with tuple objects.


In [ ]:
print("The 'mask2d' array:\n", mask2d)

print("\n-- Extracting indices of True elements in mask with np.where():")
idxs_tup = np.where(mask2d)
print("np.where(mask2d):", idxs_tup)

print("\n-- Indexing the mask itself with these indices:")
print("mask2d[np.where(mask2d)]:", mask2d[idxs_tup])

print("\n-- Extracting indices of True elements in mask with np.argwhere():")
idxs = np.argwhere(mask2d)
print("np.argwhere(mask2d):\n", idxs)

print("\n-- Indexing the mask itself with these indices:")
print("mask2d[np.where(mask2d)]:", mask2d[idxs[:,0], idxs[:,1]])

The 'mask2d' array:
 [[False  True  True]
 [False  True False]]

-- Extracting indices of True elements in mask with np.where():
np.where(mask2d): (array([0, 0, 1]), array([1, 2, 1]))

-- Indexing the mask itself with these indices:
mask2d[np.where(mask2d)]: [ True  True  True]

-- Extracting indices of True elements in mask with np.argwhere():
np.argwhere(mask2d):
 [[0 1]
 [0 2]
 [1 1]]

-- Indexing the mask itself with these indices:
mask2d[np.where(mask2d)]: [ True  True  True]


The reverse conversion (*coordinate → mask*) can be implemented by the same principle. We can write `True` values into an empty (all-false) mask using *advanced indexing* with coordinate arrays. Below are examples using coordinates obtained from `np.where()` and `np.argwhere()`, respectively:


In [ ]:
print("Integer index arrays given by np.where():", idxs_tup)

print("\nInteger index arrays given by np.argwhere():\n", idxs)

print("\n-- Creating an empty mask and writing it with the output of np.where():")
new_mask = np.zeros((2, 3), dtype=np.bool_)
print("The 'new_mask' array:\n", new_mask)

new_mask[idxs_tup] = True
print("\nThe 'new_mask' array, after writing in the coordinates:\n", new_mask)

print("\n-- Creating another empty mask and writing it with the output of np.argwhere():")
new_mask2 = np.zeros_like(new_mask)
print("The 'new_mask2' array:\n", new_mask2)

new_mask2[idxs[:,0], idxs[:,1]] = True                   # equivalent to: new_mask2[tuple(idxs.T)] = True
print("\nThe 'new_mask2' array, after writing in the coordinates:\n", new_mask2)

Integer index arrays given by np.where(): (array([0, 0, 1]), array([1, 2, 1]))

Integer index arrays given by np.argwhere():
 [[0 1]
 [0 2]
 [1 1]]

-- Creating an empty mask and writing it with the output of np.where():
The 'new_mask' array:
 [[False False False]
 [False False False]]

The 'new_mask' array, after writing in the coordinates:
 [[False  True  True]
 [False  True False]]

-- Creating another empty mask and writing it with the output of np.argwhere():
The 'new_mask2' array:
 [[False False False]
 [False False False]]

The 'new_mask2' array, after writing in the coordinates:
 [[False  True  True]
 [False  True False]]


**Note:** Besides the use above, `np.where()` can also be used in another way by providing three arguments:

```
result = np.where(condition, source1, source2)
```
In this form, `np.where()` behaves somewhat like an elementwise maximum such as `np.maximum()`, but it is more general: we can provide an arbitrary `condition` mask, and for each position the `True`/`False` value determines whether the corresponding element is taken from `source1` or `source2`. Where the mask is true, the element from `source1` is selected; where it is false, the element from `source2` is selected.

As an example, consider `result = np.maximum(source1, source2)`: we can emulate its logic with `np.where()` as follows.

```
result = np.where(source1 > source2, source1, source2)
```

As we are used to in NumPy, the three parameters of the function do not need to have the same shape here either: it is sufficient if they can be converted to a common shape based on the rules of broadcasting.

## Indexing arrays using conditions


In the first part of the chapter we discussed how to obtain masks—most often as the result of boolean operations—and how to index arrays with masks.

In many cases it is convenient to combine these two steps into a single statement. For example, suppose we want to extract the even numbers from an array. The cell below shows a loop-based Python implementation for Python lists:


In [ ]:
input_list = [7,12,-5,36,8,9,-4,0,12]

result_list = []
for elem in input_list:
  if elem % 2 == 0:
    result_list.append(elem)

print("Even numbers:", result_list)

Even numbers: [12, 36, 8, -4, 0, 12]


In NumPy, the same idea (on an array rather than a list) can be expressed much more compactly:

```
result_arr = input_arr[input_arr % 2 == 0]
```

Let’s break the evaluation of this statement into separate steps:


In [ ]:
input_arr = np.array(input_list, dtype=np.int32)
print("input_arr:", input_arr)

print("\n-- Computing mod 2 of elements:")
print("input_arr % 2:", input_arr % 2)

print("\n-- Finding which elements are even (mod 2 == 0):")
print("input_arr % 2 == 0:", input_arr % 2 == 0)

print("\n-- Getting the even elements:")
result_arr = input_arr[input_arr % 2 == 0]
print("input_arr[input_arr % 2 == 0]:", result_arr)

input_arr: [ 7 12 -5 36  8  9 -4  0 12]

-- Computing mod 2 of elements:
input_arr % 2: [1 0 1 0 0 1 0 0 0]

-- Finding which elements are even (mod 2 == 0):
input_arr % 2 == 0: [False  True False  True  True False  True  True  True]

-- Getting the even elements:
input_arr[input_arr % 2 == 0]: [12 36  8 -4  0 12]


First (after creating the array) we compute the remainder modulo 2 for each element using the `%` operator (equivalent function: `np.mod()`). This yields an integer array of the same shape as the input, since `%` is applied elementwise. Next we check which remainders are equal to zero using the `==` operator (equivalent function: `np.equal()`), which is also elementwise, but produces a boolean result array: a mask. This mask marks the even elements and can therefore be used to index the original array. The result of mask-based indexing is a one‑dimensional array containing the even numbers.

So “indexing by condition” is not a new form of *advanced indexing*: we are still doing boolean mask indexing, but we do not assign the mask itself as a value to a named Python variable after creation, we only create it temporarily within the instruction, so it can be discarded immediately after use.

Conditions can also be chained with logical operations, as shown in the cell below. By conjuncting, disjuncting, or negating the masks generated according to each condition, new masks can be created that express complex conditions:

In [ ]:
input_arr = np.array(input_list, dtype=np.int32)
print("input_arr:", input_arr)

print("\n-- Getting even OR non-negative elements:")
print("input_arr[(input_arr % 2 == 0) | (input_arr >= 0)]:", input_arr[(input_arr % 2 == 0) | (input_arr >= 0)])

print("\n-- Getting even AND non-negative elements:")
print("input_arr[(input_arr % 2 == 0) & (input_arr >= 0)]:", input_arr[(input_arr % 2 == 0) & (input_arr >= 0)])

print("\n-- Getting even AND negative elements:")
print("input_arr[(input_arr % 2 == 0) & ~(input_arr >= 0)]:", input_arr[(input_arr % 2 == 0) & ~(input_arr >= 0)])


input_arr: [ 7 12 -5 36  8  9 -4  0 12]

-- Getting even OR non-negative elements:
input_arr[(input_arr % 2 == 0) | (input_arr >= 0)]: [ 7 12 36  8  9 -4  0 12]

-- Getting even AND non-negative elements:
input_arr[(input_arr % 2 == 0) & (input_arr >= 0)]: [12 36  8  0 12]

-- Getting even AND negative elements:
input_arr[(input_arr % 2 == 0) & ~(input_arr >= 0)]: [-4]


Elementwise boolean functions can also be combined with boolean *vector → scalar* functions. In the example below, we use them to select rows and columns of a matrix that contain at least one non‑negative number, or where all numbers are non‑negative:


In [ ]:
my_matrix = np.array([[-1, -3, 2], [5, 6, 7], [-4, -9, -1]], dtype=np.int32)
print("The my_matrix array:\n", my_matrix)

print("\n-- Getting non-negative elements:")
print("my_matrix[my_matrix >= 0]:", my_matrix[my_matrix >= 0])

print("\n-- Getting columns with any non-negative elements:")
print("my_matrix[np.any(my_matrix >= 0, axis=0)], axis=0):\n", my_matrix[np.any(my_matrix >= 0, axis=0)])
print("\n-- Getting columns with non-negative elements only:")
print("my_matrix[np.all(my_matrix >= 0, axis=0)], axis=0):\n", my_matrix[np.all(my_matrix >= 0, axis=0)])

print("\n-- Getting rows with any non-negative elements:")
print("my_matrix[np.any(my_matrix >= 0, axis=1)]:\n", my_matrix[np.any(my_matrix >= 0, axis=1)])
print("\n-- Getting rows with non-negative elements only:")
print("my_matrix[np.all(my_matrix >= 0, axis=1)]:\n", my_matrix[np.all(my_matrix >= 0, axis=1)])

The my_matrix array:
 [[-1 -3  2]
 [ 5  6  7]
 [-4 -9 -1]]

-- Getting non-negative elements:
my_matrix[my_matrix >= 0]: [2 5 6 7]

-- Getting columns with any non-negative elements:
my_matrix[np.any(my_matrix >= 0, axis=0)], axis=0):
 [[-1 -3  2]
 [ 5  6  7]
 [-4 -9 -1]]

-- Getting columns with non-negative elements only:
my_matrix[np.all(my_matrix >= 0, axis=0)], axis=0):
 []

-- Getting rows with any non-negative elements:
my_matrix[np.any(my_matrix >= 0, axis=1)]:
 [[-1 -3  2]
 [ 5  6  7]]

-- Getting rows with non-negative elements only:
my_matrix[np.all(my_matrix >= 0, axis=1)]:
 [[5 6 7]]


## Generating random numbers


We can generate random numbers one by one, or generate arrays filled with random numbers, using the functions in the `np.random` module.

We can generate integers from the interval [`low`, `high`) with `np.random.randint(low, high)`. The `low` argument can be omitted, in which case sampling is done from [0, `high`). If we explicitly provide a shape tuple via the optional `size` parameter, the function creates an array of that shape and fills it with random integers. If we do not provide `size`, we get a single scalar.


In [ ]:
print("np.random.randint(10):", np.random.randint(10))   # scalar between 0 and 9
print("np.random.randint(15, 20):", np.random.randint(15, 20))   # scalar between 15 and 19

print("np.random.randint(15, 20, size=(2, 2)):\n", np.random.randint(15, 20, size=(2, 2)))

np.random.randint(10): 8
np.random.randint(15, 20): 18
np.random.randint(15, 20, size=(2, 2)):
 [[18 16]
 [19 18]]


A common use of `np.random` is generating random masks, where we place `True` and `False` values into a boolean array with a specified probability. For example, the *dropout* operation commonly used in machine learning is implemented with such masks. There are multiple ways to create them:
- `np.random.rand()` generates floating‑point numbers from [0, 1) into an array of a specified shape. The shape is provided as separate arguments (unpacked from the tuple). Given the generated array `r` and a number 0 ≤ p ≤ 1, `r < p` creates a mask where `True` appears with probability p.
- `np.random.choice()` randomly selects a specified number of elements from an array (or from an integer range). We can manually assign probabilities to each element of the input array or range, thus specifying the probability of true and false elements. On the one hand, we can choose logical values from a list/array of [False, True] or the corresponding [0, 1], or we can consider the interval [0, 2) instead of the array by specifying the number 2 as an argument.

Let's generate a random mask of shape (2, 3) using the above methods, in which true values appear with a probability of 30%!

In [ ]:
mask1 = np.random.rand(2, 3) < 0.3

print("The 'mask1' array:\n", mask1)

mask2 = np.random.choice([True, False], size=(2, 3), p=[0.3, 0.7])
print("\nThe 'mask2' array:\n", mask2)

mask3 = np.random.choice(2, size=(2, 3), p=[0.7, 0.3]).astype(np.bool_)   # 0 -> False, 1 -> True
print("\nThe 'mask3' array:\n", mask3)

The 'mask1' array:
 [[False False  True]
 [ True  True False]]

The 'mask2' array:
 [[False False False]
 [False False  True]]

The 'mask3' array:
 [[ True False  True]
 [False  True False]]


With `np.random.choice()`, the `replace` parameter controls whether sampling is done with replacement. If we call the function with `replace=False`, each element from the source can be selected at most once, and the number of generated elements cannot exceed the number of source elements.

It is also worth mentioning `np.random.shuffle()` and `np.random.permutation()`. `np.random.shuffle()` shuffles a 1D array *in place* (it returns nothing and modifies the input). For multidimensional arrays it shuffles the slices along the first axis. `np.random.permutation()` follows the same idea, but it does not modify the input array; instead, it returns a new permuted array. Instead of an array or list, we can also specify an integer `n`, in which case the function returns a randomly shuffled `np.arange(n)` array. The returned array can be then used to shuffle other arrays using advanced indexing. This approach is especially useful when we want to shuffle multiple arrays using the same permutation.

Some other important functions of the `np.random` module:
- `np.random.binomial()`: Generates random numbers from a binomial distribution.
- `np.random.normal()`: Generates random numbers from a normal/Gaussian distribution.
- `np.random.uniform()`: Generates random numbers from a uniform distribution. This is equivalent to a properly scaled and shifted call to `np.random.rand()`.
- `np.random.seed()`: Sets the random generator to a given state for reproducible results.


## Vectorizing arbitrary Python code


NumPy provides ways to apply arbitrary Python functions in a vectorized manner, similarly to NumPy ufuncs. The functions `np.apply_along_axis()`, `np.vectorize()`, and `np.frompyfunc()` can be used for this.

<font color='red'>**Using these functions should be considered carefully; in most cases they can be replaced by much more efficient solutions!**</font>

With these functions, we can apply arbitrary Python functions to slices of (possibly multidimensional) arrays independently, with a single call. So if we have a Python function written to operate on scalars, and we want to apply it to every element of an array, this may seem like a good option. Similarly, a function that operates on 1D arrays can be applied to 1D slices of a multidimensional array with one call, and the results can be concatenated automatically.

If we look into the documentation of `np.vectorize()`, we find the following lines:

_"The `vectorize` function is provided primarily for convenience, not for performance. The implementation is essentially a for loop."_

In other words, **`np.vectorize()` is essentially syntactic sugar that hides a loop of function calls and concatenation of results.** If we are aware of this fact, using `np.vectorize()` and related functions can still be justified for convenience, but the solution can be severely inefficient. When we vectorize a custom scalar Python function this way, **our code is still executed by the Python interpreter, often repeatedly.**

Of course, we can also pass NumPy functions to `np.vectorize()`, which may lead to somewhat better constructions. However, the point of vectorized programming is to perform each operation on as much data as possible with a single call, and most of the operations we use already support vectorized execution. For example, if we want to sort the elements within each row of a matrix independently, we could pass `np.sort` to `np.vectorize()` to apply it to each row. But `np.sort` already supports this directly via the `axis` parameter—without `np.vectorize()`.

For some NumPy functions that operate only on 1D arrays, or for certain Python built-ins that do not accept arrays (or have more efficient scalar implementations), `np.vectorize()`-style utilities can be useful. Still, their performance can almost always be beaten by writing true vectorized NumPy code by hand.
